# 05 · Jinja & macros

Every dbt model is a **Jinja template that renders to SQL**. `ref()` and
`source()` are macros; so are materializations. This notebook pokes at the
rendering machinery directly.

Companion reading: [docs/09](../docs/09_jinja_and_macros.md).

In [ ]:
# --- Standard setup (identical in every notebook) ---------------------------
import os, sys, json, subprocess
from pathlib import Path
from datetime import date, timedelta

from dotenv import load_dotenv

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
load_dotenv(REPO / ".env")

os.chdir(REPO / "jaffle_shop")          # dbt must run from the project dir
os.environ.setdefault("DBT_PROFILES_DIR", str(REPO / "jaffle_shop"))

from dbt.cli.main import dbtRunner

def run_dbt(args):
    """Run dbt programmatically (same engine as the CLI). Returns a
    dbtRunnerResult: .success says how it went, .result has per-node details.
    Crucially it never sys.exit()s -- perfect for notebooks."""
    print(f"$ dbt {' '.join(args)}")
    res = dbtRunner().invoke(args)
    print(f"-> success={res.success}")
    return res

def load_day(*flags):
    """Invoke the raw-data generator (plays the role of the EL tool)."""
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "generate_data.py"), *flags],
        check=True,
    )


In [ ]:
# --- Warehouse connection for %%sql cells (jupysql) and pandas --------------
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBT_USER', 'dbt')}:{os.getenv('DBT_PASSWORD', 'dbt')}"
    f"@{os.getenv('DBT_HOST', 'localhost')}:{os.getenv('DBT_PORT', '5432')}"
    f"/{os.getenv('DBT_DBNAME', 'jaffle_shop')}"
)

%load_ext sql
%sql engine
%config SqlMagic.displaylimit = 25


## 1. `dbt compile --inline`: a Jinja playground

Compile any snippet without creating a model. Perfect for testing an
expression before it goes into a file.

In [ ]:
res = run_dbt(["compile", "--inline",
    "select '{{ target.name }}' as target_name, "
    "{{ cents_to_dollars('1234') }} as amount, "
    "{{ dbt.current_timestamp() }} as compiled_at"])
assert res.success

print(res.result.results[0].node.compiled_code)

Three things rendered: the `target` context variable, this project's
`cents_to_dollars` macro, and a cross-database macro from dbt core.

## 2. Jinja loops that write SQL for you

`int_order_payments` pivots payment methods into columns with a `for` loop
over `dbt_utils.get_column_values(...)` -- compare source and compiled:

In [ ]:
print("--- SOURCE (Jinja) ---")
print(Path("models/intermediate/int_order_payments.sql").read_text())

In [ ]:
res = run_dbt(["compile", "--select", "int_order_payments"])
print("--- COMPILED (SQL) ---")
print(res.result.results[0].node.compiled_code)

Add a row to `seeds/payment_methods.csv`, `dbt seed`, recompile -- new
column, zero SQL edited.

## 3. Target-aware logic

`stg_events` embeds `limit_in_dev(50000)`: a LIMIT that only appears in the
dev target when a var asks for it. Compile both ways and diff the tail:

In [ ]:
res = run_dbt(["compile", "--select", "stg_events"])
print("normal:  ...", res.result.results[0].node.compiled_code[-120:].strip())

res = run_dbt(["compile", "--select", "stg_events", "--vars", "{limit_dev_rows: true}"])
print("limited: ...", res.result.results[0].node.compiled_code[-120:].strip())

## 4. `run-operation`: macros as admin commands

`drop_old_relations` scans the target schemas for relations the project no
longer manages (leftovers from renamed/deleted models). Dry-run by default.

In [ ]:
res = run_dbt(["run-operation", "drop_old_relations"])
assert res.success

## 5. Hooks in action: the audit log

`dbt_project.yml` wires two macros into `on-run-start` / `on-run-end`; every
dbt invocation in this sandbox has been writing a row here (including all
of yours today):

In [ ]:
%%sql
select target_name, command, node_count, ok_count, error_count,
       date_trunc('second', finished_at) as finished_at
from audit.dbt_run_log
order by finished_at desc
limit 10

## 6. The gotcha you will hit someday

**SQL comments are still Jinja.** A comment like `-- see the config() docs`
executes `config()`; writing Jinja delimiters inside a comment can break or
even silently alter the model (this repo's `dim_customers.sql` and
`int_order_payments.sql` carry war-story comments -- building this sandbox
hit both). When you need literal braces, use `{% raw %}...{% endraw %}`.